In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_val_predict
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.base import clone

from xgboost import XGBRegressor

In [3]:
riders = pd.read_csv("../dataset/processed/riders_wt.csv")
riders

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,...,FM_Ratio,WT_Ratio,LM_Ratio,O2A_Target,FM_Target,WT_Target,LM_Target,O2A_Pred,FM_Pred,WT_Pred
0,0xcdcd,DEHRES17DEL01,36.0,4.2,30.327968,78.046106,30.397968,78.116106,2022-02-12,21:55,...,0.199705,0.229905,0.244691,14.982155,9.186442,10.575639,11.255764,15.240974,10.747746,10.747746
1,0xd987,KOCRES16DEL01,21.0,4.7,10.003064,76.307589,10.043064,76.347589,2022-02-13,14:55,...,0.164921,0.368097,0.239112,5.241003,3.793185,8.466236,5.499576,5.330074,8.823434,8.823434
2,0x2784,PUNERES13DEL03,23.0,4.7,18.562450,73.916619,18.652450,74.006619,2022-03-04,17:30,...,0.270654,0.264099,0.315591,3.142776,5.683731,5.546075,6.627417,3.658157,6.187201,6.187201
3,0xc8b6,LUDHRES15DEL02,34.0,4.3,30.899584,75.809346,30.919584,75.829346,2022-02-13,09:20,...,0.061208,0.678989,0.259804,0.000000,1.224151,13.579778,5.196071,0.000000,20.347288,20.347288
4,0xdb64,KNPRES14DEL02,24.0,4.7,26.463504,80.372929,26.593504,80.502929,2022-02-14,19:50,...,0.290595,0.204737,0.305619,8.161027,11.914375,8.394199,12.530399,7.257767,7.657623,7.657623
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45488,0x7c09,JAPRES04DEL01,30.0,4.8,26.902328,75.794257,26.912328,75.804257,2022-03-24,11:35,...,0.180394,0.215488,0.280887,10.343405,5.772617,6.895603,8.988375,10.509945,6.859972,6.859972
45489,0xd641,AGRRES16DEL01,21.0,4.6,0.000000,0.000000,0.070000,0.070000,2022-02-16,19:55,...,0.240939,0.221633,0.278855,9.308600,8.673816,7.978800,10.038784,7.346757,6.133760,6.133760
45490,0x4f8d,CHENRES08DEL03,30.0,4.9,13.022394,80.242439,13.052394,80.272439,2022-03-11,23:50,...,0.402479,0.271583,0.325938,0.000000,6.439658,4.345334,5.215008,0.000000,4.930751,4.930751
45491,0x5eee,COIMBRES11DEL01,20.0,4.7,11.001753,76.986241,11.041753,77.026241,2022-03-07,13:35,...,0.207348,0.264710,0.241173,7.456005,5.391042,6.882466,6.270487,7.506968,7.126299,7.126299


In [4]:
features = [

    "Traffic_Score",
    "Weather_Score",
    "Trip_Distance_km",
    "Vehicle_Score",
    "Workload",

    "Delivery_person_Age",
    "Delivery_person_Ratings",

    "Festival",
    "City",
    "Peak_Period",

    "Type_of_order",
    "Type_of_vehicle",

    "O2A_Pred",
    "FM_Pred",
    "WT_Pred"

]

X = riders[features]
y = riders["LM_Target"]

In [5]:
cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(exclude="object").columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

In [6]:
model = XGBRegressor(

    objective="reg:squarederror",

    n_estimators=500,

    learning_rate=0.05,

    max_depth=6,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42

)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Traffic_Score',
                                                   'Weather_Score',
                                                   'Trip_Distance_km',
                                                   'Vehicle_Score', 'Workload',
                                                   'Delivery_person_Age',
                                                   'Delivery_person_Ratings',
                                                   'O2A_Pred', 'FM_Pred',
                                                   'WT_Pred']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy...
                              feature_types=None, gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.05,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=6, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=500, n_jobs=None,
                              num_parallel_tree=None, random_state=42, ...))])

In [8]:
pred = pipeline.predict(X_test)
pred = np.clip(pred, 0, None)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print(f"MAE  : {mae:.3f}")
print(f"RMSE : {rmse:.3f}")
print(f"R²   : {r2:.4f}")

MAE  : 0.987
RMSE : 1.301
R²   : 0.8250


In [9]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_pred = cross_val_predict(
    clone(pipeline),
    X,
    y,
    cv=kf,
    method="predict",
    n_jobs=-1
)

oof_pred = np.clip(oof_pred, 0, None)

riders["FM_Pred"] = oof_pred

In [10]:
oof_pred = cross_val_predict(
    clone(pipeline),
    X,
    y,
    cv=kf,
    method="predict",
    n_jobs=-1
)

oof_pred = np.clip(oof_pred, 0, None)

riders["LM_Pred"] = oof_pred

In [11]:
pipeline.fit(X, y)

joblib.dump(
    pipeline,
    "../modelv2/lm_model.pkl"
)

riders.to_csv(
    "../dataset/processed/riders_lm.csv",
    index=False
)
